# Comparing Algorithms with AlgorithmSelection & StrategySelection

`AlgorithmSelection` runs several algorithms on the same problem, repeats
each one multiple times, and collects objective values, timings, and other
statistics.  `StrategySelection` is a convenient shortcut: give it a list of
search strategies and a shared configuration dictionary – it wraps them into
`Algorithm` objects for you.

This tutorial shows how to:
* wrap algorithms (or strategies) into a contest
* run the contest and retrieve the single best solution
* inspect the raw per‑repetition data
* produce a summary report
* compare convergence of different strategies

We’ll use the 5‑dimensional **Sphere** function and compare three classic
metaheuristics: a Genetic Algorithm, Differential Evolution, and Particle
Swarm Optimisation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import metaheuristic_designer as mhd
from metaheuristic_designer.benchmarks import Sphere
from metaheuristic_designer.algorithms.algorithm_selection import AlgorithmSelection
from metaheuristic_designer.algorithms.strategy_selection import StrategySelection

# Reproducibility
rng = mhd.check_rng(42)
repetitions = 100

## 1. Objective function

We minimise the 5‑D Sphere function.

In [ ]:
DIM = 5
objfunc = Sphere(dimension=DIM)

## 2. Build the algorithms

For brevity we use the `simple` wrappers – they return ready‑to‑run `Algorithm`
instances.  You can also assemble strategies manually.

* **GA**: tournament selection, Gaussian mutation (1 gene, σ=0.1), uniform crossover
* **DE**: `DE/rand/1` with F=0.8, Cr=0.9
* **PSO**: standard parameters (w=0.7, c1=1.5, c2=1.5)

In [ ]:
ga_algo = mhd.simple.genetic_algorithm_real(
    objfunc,
    population_size=100,
    mutation_strength=0.1,
    mutated_components=1,
    stop_condition_str="max_iterations",
    max_iterations=100,
    reporter="tqdm",
    rng=rng,
)

de_algo = mhd.simple.differential_evolution_real(
    objfunc,
    population_size=100,
    de_operator_name="DE/rand/1",
    F=0.8,
    Cr=0.9,
    stop_condition_str="max_iterations",
    max_iterations=100,
    reporter="tqdm",
    rng=rng,
)

pso_algo = mhd.simple.particle_swarm_real(
    objfunc,
    population_size=50,
    w=0.7,
    c1=1.5,
    c2=1.5,
    stop_condition_str="max_iterations",
    max_iterations=100,
    reporter="tqdm",
    rng=rng,
)

## 3. Compare algorithms with AlgorithmSelection

`AlgorithmSelection` accepts a list of `Algorithm` objects and the number of
repetitions.  It takes care of internal seeding – you don’t need to manually
create fresh random states.

In [ ]:
contest = AlgorithmSelection(
    [ga_algo, de_algo, pso_algo],
    repetitions=repetitions,
)

### 3.1 Run the contest

`optimize()` executes all algorithms `repetitions` times and returns the
single best population found across all runs.

In [ ]:
best_population = contest.optimize()

### 3.2 Best solution

The returned population is a standard `Population` object.

In [ ]:
solution, objective = best_population.best_solution()
print(f"Best overall objective: {objective:.6f}")
print("Best solution vector:", solution)

### 3.3 Raw per‑repetition data

After `optimize()` the raw data is available as `contest.raw_data` (a pandas
DataFrame).  Each row is one repetition.  Columns include:
- `repetition`, `name`, `iterations`, `evaluations`, `realtime`, `cputime`
- `best_objective`, `median_objective`, `worst_objective`

In [ ]:
print("First few rows of raw data:")
display(contest.raw_data.head())

### 3.4 Summary report

The `report()` method aggregates the raw data across repetitions for each
algorithm.  It respects the optimisation mode (minimisation) so “overall
best” is the smallest objective.

In [ ]:
report = contest.report()
display(report)

## 4. Using StrategySelection (alternative)

Instead of building `Algorithm` objects yourself, you can pass search
strategies directly.  `StrategySelection` wraps each one into an `Algorithm`
with the same settings and then behaves exactly like `AlgorithmSelection`.

Here we create the same three strategies manually (no simple wrappers) to
illustrate the pattern.  In practice you can use the `simple` module here as
well.

In [ ]:
from metaheuristic_designer.strategies import GA
from metaheuristic_designer.initializers import UniformInitializer
from metaheuristic_designer.operators import create_operator
from metaheuristic_designer.parent_selection import create_parent_selection
from metaheuristic_designer.survivor_selection import create_survivor_selection

# Build strategies (no simple API)
ga_strategy = GA(
    initializer=UniformInitializer(DIM, -10, 10, population_size=100, rng=rng),
    mutation_op=create_operator("mutation.gaussian_mutation", N=1, F=0.1, rng=rng),
    crossover_op=create_operator("crossover.uniform", rng=rng),
    parent_sel=create_parent_selection("tournament", amount=50, tournament_size=3, rng=rng),
    survivor_sel=create_survivor_selection("elitism", amount=25, rng=rng),
    mutation_prob=0.3,
    crossover_prob=0.9,
    rng=rng,
)

objfunc = Sphere(dimension=DIM)

Now provide the strategies and shared algorithm parameters.

In [ ]:
common_params = {
    "stop_condition_str": "max_iterations",
    "max_iterations": 100,
    "reporter": "tqdm",
}

strategy_contest = StrategySelection(
    objfunc,
    strategy_list=[ga_strategy, de_algo.search_strategy, pso_algo.search_strategy],
    repetitions=repetitions,
    **common_params,
)

# Run it exactly like AlgorithmSelection
best_pop = strategy_contest.optimize()
solution, objective = best_pop.best_solution()
print(f"Best overall objective (via StrategySelection): {objective:.6f}")

# Raw data and report are also available
display(strategy_contest.report())

## 5. Distribution of best objectives (optional)

The raw data from the contest contains the `best_objective` for each
repetition.  We can compare the distributions with a histogram using a
logarithmic horizontal axis to handle differences in scale.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_axisbelow(True)  # moves grid behind plot elements

# KDE with logarithmic x-axis
sns.kdeplot(data=contest.raw_data, x="best_objective", hue="name", log_scale=True, fill=True, alpha=0.3, linewidth=2, ax=ax)

# Rug (small ticks along the x-axis) to show each data point
sns.rugplot(data=contest.raw_data, x="best_objective", hue="name", height=0.05, alpha=0.7, ax=ax)

ax.set_title("Distribution of best objectives (30 repetitions)")
ax.set_xlabel("Best objective (log scale)")
ax.grid()
sns.despine()
plt.tight_layout()
plt.show()